## TEXT MINING PROJECT <font color=green> **In progress**</font>
***

The goal of this project is to fill in missing data in another dataset. Originally it was downloaded from Bloomberg and concerned the compensation of higher managament for various banks around the world. Data from bloomberg was found to be missing a lot of entries so it was decided to fill in the missing information by getting bank reports from the web and extracting it from them. End result of this project should be a dataframe that could be joined with existing one.

Below are five main steps of this work:
1. Basic preparation.
2. Defining specific keywords. 
3. Identifying which pages of the pdf's contain right information.
4. Extracting the data from beforementioned pages into a dataframe. 
5. Cleaning extracted data.
***

### 1. Preparation
* ##### Importing

In [1]:
import sys, os
import PyPDF2
import re
import pandas as pd
import pikepdf


* ##### Listing all PDF files
Firstly a list of pdfs needs to be created. For that we fetch directory of all pdfs which later is used in a loop to append the list with file report names.

In [2]:
paths = [r'C:\Users\maxim\Documents\GitHub\Remuneration-Data-Text-Mining\reports', r'C:\Users\mi.martowicz\Documents\GitHub\text-mining\Reports']

#choosing correct path
for i in paths:
    if os.path.isdir(i) == True:
        pth = i
    else:
        continue 

#taking pdf names out of path of every file
pdf_dir = os.listdir(pth)

pdf_list = []

#creating list of pdf names
for file in pdf_dir:
    pdf_list.append(file)

#decrypting pdf's
#for n in pdf_list:
#    pdf = pikepdf.open(pth + '/' + n, allow_overwriting_input=True)
#    pdf.save(pth + '/' + n)   

 ***
 ### 2. Keywords 
Here are gathered all keywords that will be used for finding the pages of interest as well as for extraction later on.

In [3]:
key_words = {'exec': ['ceo', 'chief executive', 'chief executive officer', 'director'], 
    'pay': ['commission', 'remuneration', 'salary', 'compensation', 'benefits', 'variable remuneration', 
    'fixed remuneration', 'clawback', 'emoluments', 'exceptional remuneration', 'fixed fees', 'attendance fees']}

 ***
 ### 3. Building first loop
The goal for now is to create a directory of lists that will have title of the report and page numbers which contain the information of our interest.<br>
Page numbers are appended only if the page intersects with one word from keywords regarding remuneration and executive titles

* ##### Definig variables needed for the loop

In [4]:
# for testing we will do just two pdf's so im defining new list with two file titles
test_pdf = [ pdf_list[16], pdf_list[18], pdf_list[19], pdf_list[20]]

# empty dictionary , it will store the title of pdf and pages where it found matching keywords
pdf_pages = {}

# word sets for intersecting text later in the loop
word_set_1 = set(key_words['exec'])
word_set_2 = set(key_words['pay']) 

* ##### Loop

In [5]:
for element in test_pdf:
    document = PyPDF2.PdfFileReader(pth + '/' + element, strict=False)
    # here we retrieve number of pages to enable us to iterate through them below
    no_pages = document.getNumPages()
    # defining list for page numbers
    pagenum_list=[]
    for i in range(no_pages):
        # extracting text from a page and florring the letters
        text = document.getPage(i).extractText().lower()
        # set and split allows us to later iterate throug words
        text_set = set(text.split())
        if word_set_1.intersection(text_set) and word_set_2.intersection(text_set):
            # appending list with page numbers
            pagenum_list.append(i)
    # appending dictionary of lists
    pdf_pages[element] = pagenum_list


* ##### Test
And a quick test to see if it works, here we chose Santander because it only found one page.
From the results we can quickly see that it worked perfectly. in the pdf this page is a table with remuneration
in thousands of EUR

In [6]:
print(pdf_pages)

{'HSBC 2013.pdf': [5, 6, 45, 47, 332, 333, 334, 335, 336, 342, 344, 346, 350, 354, 361, 362, 363, 372, 381, 382, 386, 387, 388, 390, 391, 392, 393, 394, 397, 399, 401, 402, 403, 407, 409], 'SANTANDER CONSUMER FINANCE SPAIN 2010.pdf': [58], 'SANTANDER CONSUMER FINANCE SPAIN 2011.pdf': [59], 'SANTANDER CONSUMER FINANCE SPAIN 2012.pdf': [61]}


In [7]:
test = PyPDF2.PdfFileReader(pth + '/' + pdf_list[18], strict=False)
text123 = test.getPage(58).extractText()
print(text123)

  
51 
 The detail, by director, of the aforementioned remune ration is as follows: 
 Thousands of Euros 

  2010  2009 

 Bylaw- 
 
 Bylaw- 
 
 

  Stipulated 
 
Attendance 
  
  Stipulated 
 
Attendance 
  
 

Directors  Emoluments
 
Fees  Total  Emoluments
 
Fees  Total 

              

Mr. Antonio Escámez Torres 
 -
 
-
 
-
 
-  -  - 

Mr. Javier San Félix García 
 -
 
-
 
-
 
-  -  - 

Ms. Magdalena Salarich 
 -
 
-
 
-
 
-  -  - 

Mr. David Turiel López 
 -
 
-
 
-
 
-  -  - 

Ms. Inés Serrano González 
 -
 
-
 
-
 
-  -  - 

Mr. Ernesto Zulueta Benito  -  -  -  -  -  - 

Mr. José Antonio Álvarez Álvarez  -  -  -  -  -  - 

Mr. José María Espí Martínez  -  -  -  -  -  - 

Mr. Juan Rodríguez Inciarte  -  -  -  -  -  - 

Mr. Luis Valero Artola (*)
  
50  1  51  25  11  36 

Mr. Paul Adrian Verburgt (*)  25  1  26  25  7  32 

 75  2  77  50  18  68 

(*)  Director who was a Board member for some months in 2010 but ceased to be a director prior to 31 December 2010. 
In 2010 the Ban

***
### 4. Extracting desirable data to dataframe

* ##### New keyword dictionary <br>
We need to differientiate between keywords regarding fixed remmuneration and additional bonuses.

In [8]:
keywords_pay = {'fixed': ['remuneration', 'salary', 'compensation', 'fixed remuneration', 'emoluments', 'fixed fees', 'attendance fees'], 
    'variable': ['benefits', 'variable remuneration', 'exceptional remuneration', 'clawback', 'commission', 'variable fees', 
    'variable remuneration', 'variable compensation', 'variable fees']}


* ##### Defining variables needed for the loop

In [9]:
#pdf_pages
#path = r'C:\Users\maxim\Documents\GitHub\text-mining\Reports/'
word_set_fix = set(keywords_pay['fixed'])
word_set_var = set(keywords_pay['variable'])

* ##### Loop


**Method for extracting tables**

Succesfull instalation of camelot. To have it working you have to install camelot AND download ghostscript. When installing ghstscript you need to copy path in the installer and later go into system enviornment variables and add them two times to the path. One time just paste it and add \bin (at the end of path) and second one paste and add \temp then click ok and restart system. <br>
Download ghostscript here: https://www.ghostscript.com/releases/gsdnld.html

In [10]:
import camelot

In [11]:
#empty dataframe which will be appended with tables later in the loop
dftext = pd.DataFrame()

Note1- camelot takes in no page only in form of string, our current set has them as list of integers.<br> 
Note2 - camelot exctracts them kinda good but at the same time we can see some strange things happening in the row index 2
Note3- camelot throws specific error when a page is a picture. This could be extremely usefull when distingushing which pdfs should be read in as pics.

**OUTINE FOR LOOP**
1. first we pull tables <br>
2. IF there are no tables then file is moved to normal text extraction, IF it throws the "page is a picture error" append the file name to a list for later. **reference** of the error can be seen when using camelot on the firs and second page of santaders report, it has scanned pages of audit notes.<br>
3. Camelot tables are easily transformed into dfs. We can append df and make a new collumn in it with year and name of the bank which is in file names. The text should also be added to rows if possible. It could be like discussed earlier which is if keyword is found extrac x words before and after it.<br> <br>
4. After camelot the extraction of text needs to be done

In [14]:
#first drqft

for title, pages in pdf_pages.items():
    #list for documents that had no tables
    notabpdfs = []
    #empty list of dataframes
    tempdfs = []
    for page in pages:
        #converting page to str for camelot read
        cpage = str(page)
        #reading in the tables in given pages
        tables = camelot.read_pdf(pth + '/' + title, pages=cpage)
        if len(tables)==0:
            notabpdfs.append(title)
        else:    
            for i in range(len(tables)):
                tempdf = tables[i].df
                #deleting last 4 chars in title which is the ".pdf" part
                tempdf['Institution_year'] = title[:-4]
                # appending temporary dfs to a list
                tempdfs.append(tempdf)
                #concatanating all temporary dfs from the page
                dfconc = pd.concat(tempdfs)
                # concatanating newest version of  dftext by tables from the page
                dftext = pd.concat([dftext, dfconc], axis = 0, ignore_index = True)

    
dftext

,0,1,2,Institution_year
0,,,,HSBC 2013
1,,Thousands of Euros,,SANTANDER CONSUMER FINANCE SPAIN 2010
2,,2010,2009,SANTANDER CONSUMER FINANCE SPAIN 2010
3,Net profit for the year attributable to the Pa...,"344,915 \n \n(16,448) \n \n1,078,253,872 \n0.3...","100,597 \n \n(44,713) \n \n913,141,551 \n0.11 ...",SANTANDER CONSUMER FINANCE SPAIN 2010
4,,Thousands of Euros,,SANTANDER CONSUMER FINANCE SPAIN 2010
5,,2010,2009,SANTANDER CONSUMER FINANCE SPAIN 2010
6,Net profit for the year attributable to the Pa...,"344,915 \n \n(16,448) \n \n1,078,253,872 \n0.3...","100,597 \n \n(44,713) \n \n913,141,551 \n0.11 ...",SANTANDER CONSUMER FINANCE SPAIN 2010
7,,Thousands of Euros,,SANTANDER CONSUMER FINANCE SPAIN 2010
8,,2010,2009,SANTANDER CONSUMER FINANCE SPAIN 2010
9,Bylaw-stipulated emoluments \nAttendance fees,75 \n2,50 \n18,SANTANDER CONSUMER FINANCE SPAIN 2010


In [13]:
pdf_pages

{'HSBC 2013.pdf': [5,
  6,
  45,
  47,
  332,
  333,
  334,
  335,
  336,
  342,
  344,
  346,
  350,
  354,
  361,
  362,
  363,
  372,
  381,
  382,
  386,
  387,
  388,
  390,
  391,
  392,
  393,
  394,
  397,
  399,
  401,
  402,
  403,
  407,
  409],
 'SANTANDER CONSUMER FINANCE SPAIN 2010.pdf': [58],
 'SANTANDER CONSUMER FINANCE SPAIN 2011.pdf': [59],
 'SANTANDER CONSUMER FINANCE SPAIN 2012.pdf': [61]}

In [14]:
#for title, pages in pdf_pages.items():
    #list for documents that had no tables
#    notabpdfs = []
    #empty list of dataframes
#    tempdfs = []
#    for page in pages:
        #converting page to str for camelot read
#        cpage = str(page)
#        #reading in the tables in given pages
#        tables = camelot.read_pdf(pth + '/' + title, pages=cpage)
#        if len(tables)==0:
#            notabpdfs.append(title)
#        else:    
#            for i in range(len(tables)):
#                tempdf = tables[i].df
#                #deleting last 4 chars in title which is the ".pdf" part
#                tempdf['Institution_year'] = title[:-4]
#                # appending temporary dataframes to a list
#                tempdfs.append(tempdf)
#                #concatanating all temporary dfs from the page
#                dfconc = pd.concat(tempdfs)
#                # concatanating newest version of  dftext by tables from the page
#                dftext = pd.concat([dftext, dfconc], axis = 0, ignore_index = True)

##### Extractin strings from pdfs

In [15]:
abc = pdf_pages['HSBC 2013.pdf']
testpages = {}
testpages['HSBC 2013.pdf'] = abc
testpages

{'HSBC 2013.pdf': [5,
  6,
  45,
  47,
  332,
  333,
  334,
  335,
  336,
  342,
  344,
  346,
  350,
  354,
  361,
  362,
  363,
  372,
  381,
  382,
  386,
  387,
  388,
  390,
  391,
  392,
  393,
  394,
  397,
  399,
  401,
  402,
  403,
  407,
  409]}

In [16]:
keywords_pay = ['remuneration', 'salary', 'compensation', 'fixed remuneration', 'emoluments', 'fixed fees', 'attendance fees', 'benefits', 'variable remuneration', 'exceptional remuneration', 'clawback', 'commission', 'variable fees', 
    'variable remuneration', 'variable compensation', 'variable fees']

insert = ' page '
remuntext_dict = {}
extracted_text = pd.DataFrame()
for title, pages in testpages.items():
    document = PyPDF2.PdfFileReader(pth + '/' + title, strict=False)
    for page in pages:
        textsplit = document.getPage(page).extractText().lower().split()
        for index, item in enumerate(textsplit):
            if item in keywords_pay:
                dictitle = (str(title) + ' page ' + str(page))
                remuntext_dict[dictitle] = textsplit[index-50:index+50]
remuntext_dict = {index: ' '.join(values) for index, values in remuntext_dict.items()}
remuntext_df = pd.DataFrame.from_dict(remuntext_dict, orient = 'index')

remuntext_df

,0
HSBC 2013.pdf page 5,not seek re-election at the agm in may. in his...
HSBC 2013.pdf page 6,as the world’s population expands there is an ...
HSBC 2013.pdf page 45,change (none) total variable pay no change max...
HSBC 2013.pdf page 47,revised to 100% of fixed pay and the maximum a...
HSBC 2013.pdf page 332,chairman of the china securities regulatory co...
HSBC 2013.pdf page 333,limited; chairman and chief executive officer ...
HSBC 2013.pdf page 334,formerly chief financial officer of credit sui...
HSBC 2013.pdf page 335,plc; president and chief operating officer of ...
HSBC 2013.pdf page 336,former appointments include: chief financial a...
HSBC 2013.pdf page 342,the following topics were given: • rbwm strate...


***
### 5. Cleaning extracted data